# Data Challenge 13 — Interpreting Logistic Regression 

**Purpose**  
Apply what you learned about logistic regression interpretation by analyzing NYC Restaurant Inspection data. 
 
You’ll practice interpreting **continuous**, **binary**, and **categorical** predictors, compute **odds ratios**, and assess model accuracy. 

**Learning Goals**
- Convert coefficients to odds ratios using `np.exp()`.  
- Interpret ORs for continuous, binary, and categorical predictors.  
- Use accuracy to assess logistic regression performance.  
- Communicate results clearly and responsibly.  

**Data:** June 1, 2025 - Nov 4, 2025 Restaurant Health Inspection

[Restaurant Health Inspection](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (Quick Links)**
- LogisticRegression — https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html  
- accuracy_score — https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html  
- OneHotEncoder — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html  
- StandardScaler — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html  
- np.exp — https://numpy.org/doc/stable/reference/generated/numpy.exp.html  

**Pseudocode Plan**

1️⃣ Load cleaned restaurant inspection data from the previous challenge.  
2️⃣ Define target = `IS_A` (1 = Grade A, 0 = otherwise).  
3️⃣ Predictors →  
    • Continuous = `SCORE`  
    • Binary = `CRITICAL_NUM`  
    • Categorical = `BORO`  
4️⃣ Scale continuous variables; encode categorical ones.  
5️⃣ Fit `LogisticRegression`.  
6️⃣ Exponentiate coefficients (np.exp()) → odds ratios.  
7️⃣ Interpret one continuous, one binary, and one categorical coefficient.  
8️⃣ Evaluate accuracy.  
9️⃣ Reflect on scaling choices and communication of odds.  


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

## Step 1 — Imports and Plot Defaults

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import seaborn as sns
from sklearn import preprocessing


### Step 2 — Load CSV, Create Columns, Preview

- Point to your New York City Restaurant Inspection Data 
- Create the `is_A` and `critical_num` columns like you did in L11 notebook

In [21]:


df = pd.read_csv("/Users/Marcy_Student/Downloads/DOHMH_New_York_City_Restaurant_Inspection_Results_20251104 copy.csv")

# Making numeric
df['SCORE'] = pd.to_numeric(df['SCORE'])

# Dropping NA
df_cleanscore = df[df['SCORE'].notna()].copy()

# Creating target variable for Grade A
df_cleanscore['IS_A'] = (df_cleanscore['GRADE'] == 'A').astype(int)

# Binary flag for critical violations
df_cleanscore['CRITICAL_NUM'] = (df_cleanscore['CRITICAL FLAG'] == 'Critical').astype(int)

display(df_cleanscore.shape)
display(df_cleanscore.info())


(274939, 29)

<class 'pandas.core.frame.DataFrame'>
Index: 274939 entries, 18 to 291277
Data columns (total 29 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  274939 non-null  int64  
 1   DBA                    274939 non-null  object 
 2   BORO                   274939 non-null  object 
 3   BUILDING               274150 non-null  object 
 4   STREET                 274939 non-null  object 
 5   ZIPCODE                272033 non-null  float64
 6   PHONE                  274934 non-null  object 
 7   CUISINE DESCRIPTION    274939 non-null  object 
 8   INSPECTION DATE        274939 non-null  object 
 9   ACTION                 274939 non-null  object 
 10  VIOLATION CODE         273397 non-null  object 
 11  VIOLATION DESCRIPTION  273397 non-null  object 
 12  CRITICAL FLAG          274939 non-null  object 
 13  SCORE                  274939 non-null  float64
 14  GRADE                  142194 non-null  

None

In [22]:
df.columns

Index(['CAMIS', 'DBA', 'BORO', 'BUILDING', 'STREET', 'ZIPCODE', 'PHONE',
       'CUISINE DESCRIPTION', 'INSPECTION DATE', 'ACTION', 'VIOLATION CODE',
       'VIOLATION DESCRIPTION', 'CRITICAL FLAG', 'SCORE', 'GRADE',
       'GRADE DATE', 'RECORD DATE', 'INSPECTION TYPE', 'Latitude', 'Longitude',
       'Community Board', 'Council District', 'Census Tract', 'BIN', 'BBL',
       'NTA', 'Location'],
      dtype='object')

## Step 3 — Define Predictors & Target

- Target is `is_A` 
- X predictors are: SCORE, CRITICAL_NUM (created in Step 2), BORO


In [23]:
testdf = df_cleanscore[['IS_A', 'SCORE', 'CRITICAL_NUM', 'BORO']]

y = testdf['IS_A']
X = testdf[['SCORE', 'CRITICAL_NUM', 'BORO']]


## Step 4 — Split Data (70/30 Stratify by Target)

In [ ]:
np.random.seed(16)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=16, stratify=y)

## Step 5 – Preprocessing (You can chose to do this in a Pipeline)  

- Scale continuous features  
- Pass binary as is  
- One-hot encode categorical feature (`BORO`)  

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), ['BORO'])
    ],
    remainder='passthrough'
)

model = Pipeline(
    steps=[
        ('processing', preprocessor),
        ('model', LogisticRegression())
    ]
)

## Step 6 – Fit Model & Evaluate Accuracy

- Fit `is_A ~ score` using **LogisticRegression**  
- Compute predictions with `.predict()`  
- Evaluate accuracy with `accuracy_score()`

In [ ]:
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, preds), 5))

Accuracy: 0.95187


## Step 7 – Extract Coefficients and Convert to Odds Ratios


In [27]:
names = preprocessor.get_feature_names_out()
coefficients = model.named_steps['model'].coef_[0]
oddsratio = np.exp(coefficients)

info = pd.DataFrame({
    'columns': names,
    'coefficients': coefficients,
    'oddsratio': oddsratio
})

info


,columns,coefficients,oddsratio
0,categorical__BORO_Bronx,0.750595,2.118260
1,categorical__BORO_Brooklyn,0.767776,2.154968
2,categorical__BORO_Manhattan,0.799066,2.223464
3,categorical__BORO_Queens,0.884105,2.420818
4,categorical__BORO_Staten Island,0.840322,2.317114
5,remainder__SCORE,-0.323869,0.723345
6,remainder__CRITICAL_NUM,-0.099690,0.905118


## Step 8 – Interpret Each Predictor 

**Remember**
💡 OR > 1 → increases odds of Grade A  
💡 OR < 1 → decreases odds of Grade A

**Type markdown interpreting all 3 predictors in plain english**


SCORE 
With an odds ratio of 0.72, every additional inspection point lowers the odds of getting an A by about 28%. Higher scores mean worse inspections, so they reduce the chance of earning an A.

CRITICAL_NUM 
Restaurants with a critical violation have odds of receiving an A that are about 0.91 times those without one, meaning they are slightly less likely to get an A.

BORO 
Restaurants in Queens have odds of receiving an A that are about 2.42 times higher than restaurants in the baseline for borough.

# We Share — Reflection & Wrap-Up

Write **one short paragraphs** (4–6 sentences). Be specific and use evidence from your notebook.

**Which predictor had the strongest relationship with getting an A grade?**  
Use the odds ratios and accuracy to support your answer.  

Borough has the strongest relationship with whether a restaurant receives an A grade. Queens, in particularly shows the highest odds ratio of 2.42, showing that restaurants there are far more likely to earn an A than those in the reference borough. Compared to BORO, SCORE and CRITICAL_NUM have a less impactful relationship. The model also performs well overall, with an accuracy of about 0.95, suggesting that the selected predictors do a solid job at detect A-grade restaurants from others. Overall, borough differences seem to play the largest role in shaping inspection outcomes in this dataset.